### **1. Handling Missing Values**

- Load the titanic_split dataset
- Explore Missing Values
- Demonstrate why dropna is not always a good idea
- How to detect columns with high missing percentage
- Fill missing values without data leakage.
    - Median for Numeric columns using Train only.
    - Mode for categorical columns using Train only.
    - Apply the same values to TRAIN and TEST.
- Save the imputed train and test sets.

In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns',None)

### **2. Load Split Dataset**

In [3]:
train_df = pd.read_csv("titanic_train_raw.csv")
test_df = pd.read_csv("titanic_test_raw.csv")

print("Train Shape:",train_df.shape)
print("Test Shape:",test_df.shape)

Train Shape: (712, 14)
Test Shape: (179, 14)


**3. Seperate X and y**

In [4]:
target = "survived"

In [6]:
# training set - Features and target Variable
X_train = train_df.drop(columns= [target])
y_train = train_df[target]

X_test = test_df.drop(columns= [target])
y_test = test_df[target]

In [8]:
X_test.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,24.0,2,0,24.1500,S,Third,man,True,NaN,Southampton,False
1,3,male,44.0,0,1,16.1000,S,Third,man,True,NaN,Southampton,False
2,3,male,22.0,0,0,7.2250,C,Third,man,True,NaN,Cherbourg,True
3,3,male,41.0,2,0,14.1083,S,Third,man,True,NaN,Southampton,False
4,3,female,NaN,1,0,15.5000,Q,Third,woman,False,NaN,Queenstown,False


In [9]:
y_test.head()

0    0
1    0
2    1
3    0
4    1
Name: survived, dtype: int64

### **4. Checking Missing Values before processing**

In [11]:
print("Missing values in TRAIN(before)")
print(X_train.isna().sum())

print("-"* 100)

print("Missing values in TEST(before)")
print(X_test.isna().sum())

Missing values in TRAIN(before)
pclass           0
sex              0
age            137
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           553
embark_town      2
alone            0
dtype: int64
----------------------------------------------------------------------------------------------------
Missing values in TEST(before)
pclass           0
sex              0
age             40
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
deck           135
embark_town      0
alone            0
dtype: int64


In [12]:
print("Missing values percentage in TRAIN(before)")
print(X_train.isna().mean()*100)

print("-"* 100)

print("Missing values perentage in TEST(before)")
print(X_test.isna().mean()*100)

Missing values percentage in TRAIN(before)
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64
----------------------------------------------------------------------------------------------------
Missing values perentage in TEST(before)
pclass          0.000000
sex             0.000000
age            22.346369
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.000000
class           0.000000
who             0.000000
adult_male      0.000000
deck           75.418994
embark_town     0.000000
alone           0.000000
dtype: float64


### **5. Demonstration - Why dropna(dropping the rows) is not generally used in ML pipeline**

In [14]:
print("Demonstraition of dropna:")
print("Original TRAIN shape",X_train.shape)

dropped_demo = X_train.dropna()
print("Shape after dropna:",dropped_demo.shape)

Demonstraition of dropna:
Original TRAIN shape (712, 13)
Shape after dropna: (141, 13)


### **6. Demonstration: Columns with high missing percentage**
- If your dataset has a column with more than 40% of missing values it's better to drop that column.

In [15]:
print("Missing values percentage in TEST(before)")
print(X_test.isna().mean() * 100)

print("Missing values percentage in TRAIN(before)")
print(X_train.isna().mean()*100)

print("-"* 100)

print("Missing values perentage in TEST(before)")
print(X_test.isna().mean()*100)

Missing values percentage in TEST(before)
pclass          0.000000
sex             0.000000
age            22.346369
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.000000
class           0.000000
who             0.000000
adult_male      0.000000
deck           75.418994
embark_town     0.000000
alone           0.000000
dtype: float64
Missing values percentage in TRAIN(before)
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64
----------------------------------------------------------------------------------------------------
Missing values perentage in TEST(before)
pclass          0.000000
sex             0.000000
age            22.346369
sibsp           0.

In [19]:
missing_threshold = 40

missing_percent_train = X_train.isna().mean() * 100
print("Percent missing in each column")
print(missing_percent_train)

high_missing_cols = missing_percent_train[missing_percent_train > 40]


Percent missing in each column
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64


In [ ]:
# X_train = X_train.drop(columns= high_missing_cols)
# X_test = X_test.drop(columns= high_missing_cols)

In [20]:
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

In [21]:
X_train_imputed.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,NaN,0,0,56.4958,S,Third,man,True,NaN,Southampton,True
1,2,male,NaN,0,0,0.0000,S,Second,man,True,NaN,Southampton,True
2,1,male,NaN,0,0,221.7792,S,First,man,True,C,Southampton,True
3,3,female,18.0,0,1,9.3500,S,Third,woman,False,NaN,Southampton,False
4,2,female,31.0,1,1,26.2500,S,Second,woman,False,NaN,Southampton,False


In [22]:
X_train_imputed.drop(columns=["embark_town"])

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,alone
0,3,male,NaN,0,0,56.4958,S,Third,man,True,NaN,True
1,2,male,NaN,0,0,0.0000,S,Second,man,True,NaN,True
2,1,male,NaN,0,0,221.7792,S,First,man,True,C,True
3,3,female,18.0,0,1,9.3500,S,Third,woman,False,NaN,False
4,2,female,31.0,1,1,26.2500,S,Second,woman,False,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...
707,3,female,NaN,0,0,7.8792,Q,Third,woman,False,NaN,True
708,1,female,35.0,0,0,512.3292,C,First,woman,False,NaN,True
709,3,female,48.0,1,3,34.3750,S,Third,woman,False,NaN,False
710,1,male,47.0,0,0,38.5000,S,First,man,True,E,True


In [23]:
X_train_imputed.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,NaN,0,0,56.4958,S,Third,man,True,NaN,Southampton,True
1,2,male,NaN,0,0,0.0000,S,Second,man,True,NaN,Southampton,True
2,1,male,NaN,0,0,221.7792,S,First,man,True,C,Southampton,True
3,3,female,18.0,0,1,9.3500,S,Third,woman,False,NaN,Southampton,False
4,2,female,31.0,1,1,26.2500,S,Second,woman,False,NaN,Southampton,False


In [26]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["pclass", "sex", "embarked", "class", "who", "adult_male", "deck", "embark_town", "alone"]

print("Numerical Columns:",num_cols)
print("Categorical Columns:", cat_cols)

Numerical Columns: ['age', 'sibsp', 'parch', 'fare']
Categorical Columns: ['pclass', 'sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']


### **7. Impute missing values the right way(TRAIN ONLY)**

In [24]:
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

**7.1 Numeric Imputation**

In [27]:
numeric_medians = {}

for col in num_cols:
    median_val = X_train_imputed[col].median()
    numeric_medians[col] = median_val
    print(f"Filling numeric column {col} with TRAIN Median:{median_val}")
    X_train_imputed[col] = X_train_imputed[col].fillna(median_val)
    X_test_imputed[col] = X_test_imputed[col].fillna(median_val)

Filling numeric column age with TRAIN Median:28.5
Filling numeric column sibsp with TRAIN Median:0.0
Filling numeric column parch with TRAIN Median:0.0
Filling numeric column fare with TRAIN Median:14.4542


**7.2 Categorical Imputation**

In [32]:
categorical_modes = {}

for col in cat_cols:
    mode_val = X_train_imputed[col].mode().iloc[0]
    categorical_modes[col] = mode_val
    print(f"Filling categorical column {col} with TRAIN Mode:{mode_val}")
    X_train_imputed[col] = X_train_imputed[col].fillna(mode_val)
    X_test_imputed[col] = X_test_imputed[col].fillna(mode_val)

Filling categorical column pclass with TRAIN Mode:3
Filling categorical column sex with TRAIN Mode:male
Filling categorical column embarked with TRAIN Mode:S
Filling categorical column class with TRAIN Mode:Third
Filling categorical column who with TRAIN Mode:man
Filling categorical column adult_male with TRAIN Mode:True
Filling categorical column deck with TRAIN Mode:C
Filling categorical column embark_town with TRAIN Mode:Southampton
Filling categorical column alone with TRAIN Mode:True


In [34]:
print("Missing values in TRAIN(after)")
print(X_train_imputed.isna().sum())

print("-"* 100)

print("Missing values in TEST(after)")
print(X_test_imputed.isna().sum())

Missing values in TRAIN(after)
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alone          0
dtype: int64
----------------------------------------------------------------------------------------------------
Missing values in TEST(after)
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alone          0
dtype: int64


**8. Recombine features and Target and save it for future use**

In [ ]:
train_imputed_df = X_train_imputed.copy()
test_imputed_df = X_test_imputed.copy()

train_imputed_df[target] = y_train.values
test_imputed_df[target] = y_test.values

train_imputed_df.to_csv("titanic_train_imputed.csv", index= False)
test_imputed_df.to_csv("titanic_test_imputed.csv", index = False)

print("\n Saved Imputed Data")
print("- titanic_train_imputed.csv")
print("- titanic_test_imputed.csv")



 Saved Imputed Data
- titanic_train_imputed.csv
- titanic_test_imputed.csv
